### Preamble

In [7]:
import numpy as np
import pandas as pd
import pickle, os, glob, time, re
from datetime import datetime
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
os.makedirs("results/cache", exist_ok=True)
os.makedirs("results/figures", exist_ok=True)

now = lambda: datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [8]:
from traffic_scheduling.single.ppo import PpoRl
from traffic_scheduling.single.threshold import ThresholdGridSearch
from traffic_scheduling.single.imitation import ImitationLearning
from traffic_scheduling.single.basics import bimodal_exponential, generate_instance

In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/home/jeroen/repos/traffic-scheduling/single


## Specify experiments

First, specify the ingredients: distributions and scheduling models (i.e., procedures that train/eval).
Specify experiments by combining these ingredients. When the train distribution differs from the eval distribution, we are effectively measuring how well the model generalizes.

In [ ]:
N_test = 50          # number of test instances used in evaluation
timelimit_test = 60  # one minute max for optimal test schedule

models = {
    'Threshold':   ThresholdGridSearch(tau_range=np.linspace(0, 1, 20)),
    'Imitation':   ImitationLearning(max_k=5, N_train=20, timelimit=20),
    'PPO':         PpoRl(max_k=5, steps=500_000),
}

gap_distributions = {
    # these have the same arrival intensity, computed using s2 = (mu - p*s1) / (1 - p)
    'high': bimodal_exponential(p=0.5, s1=0.1, s2=10.00),
    'low':  bimodal_exponential(p=0.1, s1=0.1, s2=5.6),
}

# definition of experiments (measure generalization by providing different test (n, F) pair)
exps = [
    ([20, 20], 'low', 'Imitation',  [20, 20], 'low'),
    ([20, 20], 'high', 'Imitation', [20, 20], 'high'),
    ([20, 20], 'low', 'Threshold',  [20, 20], 'low'),
    ([20, 20], 'high', 'Threshold', [20, 20], 'high'),
    ([20, 20], 'low', 'PPO',        [20, 20], 'low'),
    ([20, 20], 'high', 'PPO',       [20, 20], 'high'),

    ([40, 40], 'low', 'Imitation',  [40, 40], 'low'),
    ([40, 40], 'high', 'Imitation', [40, 40], 'high'),
    ([40, 40], 'low', 'Threshold',  [40, 40], 'low'),
    ([40, 40], 'high', 'Threshold', [40, 40], 'high'),
    ([40, 40], 'low', 'PPO',        [40, 40], 'low'),
    ([40, 40], 'high', 'PPO',       [40, 40], 'high'),

    ([60, 60], 'low', 'Imitation',  [60, 60], 'low'),
    ([60, 60], 'high', 'Imitation', [60, 60], 'high'),
    ([60, 60], 'low', 'Threshold',  [60, 60], 'low'),
    ([60, 60], 'high', 'Threshold', [60, 60], 'high'),
    ([60, 60], 'low', 'PPO',        [60, 60], 'low'),
    ([60, 60], 'high', 'PPO',       [60, 60], 'high'),

    ([20, 20, 20], 'low', 'Imitation',  [20, 20, 20], 'low'),
    ([20, 20, 20], 'high', 'Imitation', [20, 20, 20], 'high'),
    ([20, 20, 20], 'low', 'Threshold',  [20, 20, 20], 'low'),
    ([20, 20, 20], 'high', 'Threshold', [20, 20, 20], 'high'),
    ([20, 20, 20], 'low', 'PPO',        [20, 20, 20], 'low'),
    ([20, 20, 20], 'high', 'PPO',       [20, 20, 20], 'high'),

    ([40, 40, 40], 'low', 'Imitation',  [40, 40, 40], 'low'),
    ([40, 40, 40], 'high', 'Imitation', [40, 40, 40], 'high'),
    ([40, 40, 40], 'low', 'Threshold',  [40, 40, 40], 'low'),
    ([40, 40, 40], 'high', 'Threshold', [40, 40, 40], 'high'),
    ([40, 40, 40], 'low', 'PPO',        [40, 40, 40], 'low'),
    ([40, 40, 40], 'high', 'PPO',       [40, 40, 40], 'high'),

    ([60, 60, 60], 'low', 'Imitation',  [60, 60, 60], 'low'),
    ([60, 60, 60], 'high', 'Imitation', [60, 60, 60], 'high'),
    ([60, 60, 60], 'low', 'Threshold',  [60, 60, 60], 'low'),
    ([60, 60, 60], 'high', 'Threshold', [60, 60, 60], 'high'),
    ([60, 60, 60], 'low', 'PPO',        [60, 60, 60], 'low'),
    ([60, 60, 60], 'high', 'PPO',       [60, 60, 60], 'high'),
]

### Compile

Next, we "compile" these textual experiment specifications:
 - add explicit generator function based on the textual specification,
 - instantiate each model class (optionally with parameters).

In [24]:
experiments = pd.DataFrame(exps, columns=['n_train', 'F_train', 'model_spec', 'n_test', 'F_test'])

for i, row in experiments.iterrows():
    for m in ['train', 'test']:
        n = row[f'n_{m}']; F = row[f'F_{m}']
        # when F is not a list, broadcast according to number of routes
        if isinstance(F, str): 
            F = [F]*len(n)
            row[f'F_{m}'] = F # write broadcasted list back
        F = [gap_distributions[f] for f in F]
        experiments.loc[i, f'gen_{m}'] = lambda F=F.copy(), n=n.copy(): generate_instance(F, n=n)
        experiments.loc[i, f'gen_{m}_name'] = '_'.join(str(s) for s in row[f'n_{m}']) + '_' + '_'.join(row[f'F_{m}'])
    experiments.loc[i, f'name'] = experiments.loc[i, 'gen_train_name'] + '_' + experiments.loc[i, 'model_spec'] + '_' + experiments.loc[i, 'gen_train_name']

# look up model by name, put in pandas table
experiments['model'] = experiments.apply(lambda row: models[row['model_spec']], axis=1)

experiments.drop(columns=['gen_train', 'gen_test', 'model'])

,n_train,F_train,model_spec,n_test,F_test,gen_train_name,gen_test_name,name
0,"[20, 20]","[low, low]",Imitation,"[20, 20]","[low, low]",20_20_low_low,20_20_low_low,20_20_low_low_Imitation_20_20_low_low
1,"[20, 20]","[high, high]",Imitation,"[20, 20]","[high, high]",20_20_high_high,20_20_high_high,20_20_high_high_Imitation_20_20_high_high
2,"[20, 20]","[low, low]",Threshold,"[20, 20]","[low, low]",20_20_low_low,20_20_low_low,20_20_low_low_Threshold_20_20_low_low
3,"[20, 20]","[high, high]",Threshold,"[20, 20]","[high, high]",20_20_high_high,20_20_high_high,20_20_high_high_Threshold_20_20_high_high
4,"[20, 20]","[low, low]",PPO,"[20, 20]","[low, low]",20_20_low_low,20_20_low_low,20_20_low_low_PPO_20_20_low_low
5,"[20, 20]","[high, high]",PPO,"[20, 20]","[high, high]",20_20_high_high,20_20_high_high,20_20_high_high_PPO_20_20_high_high
6,"[40, 40]","[low, low]",Imitation,"[40, 40]","[low, low]",40_40_low_low,40_40_low_low,40_40_low_low_Imitation_40_40_low_low
7,"[40, 40]","[high, high]",Imitation,"[40, 40]","[high, high]",40_40_high_high,40_40_high_high,40_40_high_high_Imitation_40_40_high_high
8,"[40, 40]","[low, low]",Threshold,"[40, 40]","[low, low]",40_40_low_low,40_40_low_low,40_40_low_low_Threshold_40_40_low_low
9,"[40, 40]","[high, high]",Threshold,"[40, 40]","[high, high]",40_40_high_high,40_40_high_high,40_40_high_high_Threshold_40_40_high_high


## Run experiments

### Compute optimal test schedules

In [25]:
def gen_test_data_and_optimal_solutions(gen, name):
    filename = f"results/cache/test-schedules-{name}.pkl"
    path = Path(filename)
    if path.is_file():
        with open(filename, "rb") as f:
            instances, time_opt = pickle.load(f)
        print("Loading test data from cache")
    else:
        print(f'Generating test data (and computing optimal schedules) {now()}')
        instances = [gen() for _ in range(N_test)]
        start_time = time.time()
        for s in tqdm(instances, desc="MILP solving"):
            s.solve(timelimit=timelimit_test, cutting_planes=[2, 3])
        time_opt = time.time() - start_time
        print(f'Done MILP solving {now()}')

        with open(filename, "wb") as file:
            pickle.dump([instances, time_opt], file)

    return instances, time_opt

### Run all experiments

In [26]:
for i, exp in experiments.iterrows():
    print(f"{now()} Experiment {i}: {exp['n_train']}, {exp['F_train']} -> {exp['model_spec']} -> {exp['n_test']}, {exp['F_test']}")

    # create test instances and solve them to optimality
    test_instances, time_opt = gen_test_data_and_optimal_solutions(exp['gen_test'], exp['gen_test_name'])
    opt = sum([t.opt.delay for t in test_instances]) / len(test_instances)
    experiments.loc[i, 'opt'] = opt
    experiments.loc[i, 'time_opt'] = time_opt

    # train model
    model = exp['model'] # ref to instantiated model
    start_time = time.time()
    model.train(exp['gen_train'], name=exp['name'])
    time_train = time.time() - start_time

    # report model training
    if hasattr(model, 'report'): model.report(exp['name'])

    # evaluate model
    start_time = time.time()
    obj = sum([model.eval(s) for s in test_instances]) / len(test_instances)
    time_test = time.time() - start_time
    
    experiments.loc[i, 'obj'] = obj
    experiments.loc[i, 'gap'] = 100 * ((obj / opt) - 1)
    experiments.loc[i, 'time_train'] = time_train
    experiments.loc[i, 'time_test'] = time_test / len(test_instances)

print(f"{now()} Done.")
os.system('notify-send "traffic-scheduling" "Experiments done"');

2025-11-08 10:41:58 Experiment 0: [20, 20], ['low', 'low'] -> Imitation -> [20, 20], ['low', 'low']
Generating test data (and computing optimal schedules) 2025-11-08 10:41:58


MILP solving: 100%|██████████| 50/50 [00:02<00:00, 17.95it/s]


Done MILP solving 2025-11-08 10:42:00


Generating expert demonstration: 100%|██████████| 20/20 [00:01<00:00, 17.36it/s]


2025-11-08 10:42:13 Experiment 1: [20, 20], ['high', 'high'] -> Imitation -> [20, 20], ['high', 'high']
Generating test data (and computing optimal schedules) 2025-11-08 10:42:13


...

2025-11-08 10:56:33 Experiment 14: [60, 60], ['low', 'low'] -> Threshold -> [60, 60], ['low', 'low']
Loading test data from cache
searching tau over 0.00, 0.05, 0.11, 0.16, 0.21, 0.26, 0.32, 0.37, 0.42, 0.47, 0.53, 0.58, 0.63, 0.68, 0.74, 0.79, 0.84, 0.89, 0.95, 1.00
2025-11-08 10:56:51 Experiment 15: [60, 60], ['high', 'high'] -> Threshold -> [60, 60], ['high', 'high']
Loading test data from cache
searching tau over 0.00, 0.05, 0.11, 0.16, 0.21, 0.26, 0.32, 0.37, 0.42, 0.47, 0.53, 0.58, 0.63, 0.68, 0.74, 0.79, 0.84, 0.89, 0.95, 1.00
2025-11-08 10:57:09 Experiment 16: [60, 60], ['low', 'low'] -> PPO -> [60, 60], ['low', 'low']
Loading test data from cache


Training: 100%|██████████| 500000/500000 [02:22<00:00, 3497.88step/s]


2025-11-08 10:59:33 Experiment 17: [60, 60], ['high', 'high'] -> PPO -> [60, 60], ['high', 'high']
Loading test data from cache


Training: 100%|██████████| 500000/500000 [02:22<00:00, 3498.31step/s]


2025-11-08 11:01:57 Done.


### Save results

In [ ]:
stamp = datetime.now().strftime("%Y-%m-%d_%H:%M:%S")
filename = f"results/experiments_{stamp}.pkl"

# remove the columns containing lambda functions
results = experiments.drop(columns=['gen_train', 'gen_test', 'model'])

with open(filename, "wb") as file:
    pickle.dump(results, file)

## Generate results table

Load results from file, picking the latest saved one by default. Change this manually if you need to use some older save, or if you want to combine results.

In [28]:
def get_latest_experiment():
    pattern = os.path.join("results", "experiments_*.pkl")
    files = glob.glob(pattern)
    if not files: raise FileNotFoundError("No saved experiment files found.")

    def extract_datetime(filepath):
        filename = os.path.basename(filepath)
        timestamp_str = filename.replace("experiments_", "").replace(".pkl", "")
        return datetime.strptime(timestamp_str, "%Y-%m-%d_%H:%M:%S")

    return max(files, key=extract_datetime)

In [122]:
# filename = get_latest_experiment()
# with open(filename, "rb") as f:
#     results = pickle.load(f)

filename = 'results/experiments_2025-11-08_03:56:47.pkl'
with open(filename, "rb") as f:
    results1 = pickle.load(f)

filename = 'results/experiments_2025-11-08_11:01:57.pkl'
with open(filename, "rb") as f:
    results2 = pickle.load(f)

results = pd.concat([results2, results1], ignore_index=True)

### Data wrangling

Remove all columns that will not appear in the final table.

In [123]:
results = results.drop(columns=['name', 'n_test', 'F_test', 'gen_train_name', 'gen_test_name'])
# general precaution: replace underscores with spaces to avoid latex errors
results.columns = results.columns.str.replace('_', ' ')

Although the above setup supports a different arrival distribution per route, we do not use this right now. To safe space in the results table, we only show 'high' or 'low' once for F_train.

In [124]:
for i, row in results.iterrows():
    col = 'n train'
    # remove spaces between the numbers
    results.loc[i, col] = r'[' + ','.join(str(i) for i in row[col]) + r']'

    col = 'F train'
    # take 'high' or 'low'
    results.loc[i, col] = row[col][0]

In [126]:
# pivot
results = results.pivot_table(
    index=['n train', 'F train'],   # rows
    columns='model spec',           # columns
    values=['opt', 'time opt', 'obj', 'gap', 'time train'],  # metrics per method
    sort=False
)

In [127]:
# swap levels of MultiIndex
results.columns = results.columns.swaplevel(0, 1)
# sort columns by model first
results = results.sort_index(axis=1, level=0)

In [128]:
# collapse the 'opt' and 'time opt' columns into new top-level column
# (because these are guaranteed to be the same for every method, by construction; and caching)
top_cols_to_collapse = results.columns.get_level_values(0).unique()
sub_cols_to_collapse = ['opt', 'time opt']

# take first column per second-level (safe because values are equal)
collapsed = results[top_cols_to_collapse].T.groupby(level=1).first().T
collapsed = collapsed[sub_cols_to_collapse]

# assign a new top-level name
collapsed.columns = pd.MultiIndex.from_product([["MILP"], collapsed.columns])

# drop original columns and concatenate new column back to the original df
results = pd.concat([results.drop(columns=sub_cols_to_collapse, level=1), collapsed], axis=1)

In [129]:
# put MILP column first
new_order = ["MILP"] + ["Threshold", "Imitation", "PPO"]
results = results.reindex(columns=new_order, level=0)

# order sub-level names
sub_level_order = ['obj', 'gap', 'time train', 'opt', 'time opt']
results = results.reindex(columns=sub_level_order, level=1)

# rename the top and bottom level
results.columns.names = ["model", "measurement"]

In [130]:
# further index and column renaming to taste
results.rename(columns={
    'time opt':  r'$T$',
    'time train': r'$T_\text{train}$',
    'n train': r'$n_r$',
    'F train': r'$F_\text{train}',
}, inplace=True)
results.index.rename([r'$n_r$', r'$F_\text{train}$'], inplace=True)

In [131]:
# convert to gaps strings
cols_x = results.columns.get_level_values(1) == 'gap'
results = results.astype({ col: str for col in results.columns[cols_x] })

def bold_min_sublevel(row):
    new_row = row.copy()
    cols_x = row.index.get_level_values(1) == 'gap'
    # find min among these columns
    min_val = row[cols_x].astype(float).min()
    # apply bold only to min in that sublevel
    new_row[cols_x] = [f"\\textbf{{{float(v):.2f}}}\\%" if float(v)==min_val else f"{float(v):.2f}\\%" 
                       for v in row[cols_x]]
    return new_row

# hightlight min gap, in each row
results = results.apply(bold_min_sublevel, axis=1)

### LaTeX table formatting

In [133]:
from traffic_scheduling.single.util import format_duration

# 2-leveled columns, so apply formatting at every columns,
# e.g. (PPO1, gap), (PPO2, gap), separately
time_cols = [r'$T$', r'$T_\text{train}$']
time_cols = results.columns[results.columns.isin(time_cols, 1)]
formatters = { col: format_duration for col in time_cols }

# latex \begin{table}{...} format
column_format = 'cc|cc|ccc|ccc|ccc'
latex_table = results.to_latex(multirow=True, column_format=column_format, multicolumn_format='c',
                               escape=False, float_format="%.2f", formatters=formatters,
                               caption="Performance Table")

In [134]:
# some further automated formatting (just search/replace)
# 1. remove all \cline
latex_table = re.sub(r'\\cline\{.*?\}\s*\n?', '', latex_table)
# 2. remove \toprule
latex_table = re.sub(r'\\toprule\s*\n?', '', latex_table)
# 3. replace \bottomrule with \midrule
latex_table = re.sub(r'\\bottomrule\s*\n?', '\\\\midrule\n', latex_table)
# 4. TODO: extract tabular
# latex_table = re.sub()
match = re.search(r"\\begin\{tabular.*?\\end\{tabular\}", latex_table, re.DOTALL)
latex_table = match.group(0)

# write complete latex template for preview
# (with timestamp)
date_line = f"\\noindent\\textit{{Compiled on: {now()}}}\\\\\n"
latex = f"""
\\documentclass{{article}}
\\usepackage{{booktabs}}
\\usepackage{{multirow}}
\\usepackage{{siunitx}}
\\usepackage[skip=10pt]{{caption}}
\\usepackage[margin=1cm]{{geometry}}
\\begin{{document}}
\\begin{{table}}
{latex_table}
\\end{{table}}
{date_line}
\\end{{document}}"""

file_prefix = 'results/experiments'

with open(file_prefix + '.preview.tex', 'w') as f:
    f.write(latex)
!tectonic {file_prefix + '.preview.tex'}

Running TeX ...
Rerunning TeX because "experiments.preview.aux" changed ...
Running xdvipdfmx ...
Writing `results/experiments.preview.pdf` (17.58 KiB)
Skipped writing 1 intermediate files (use --keep-intermediates to keep them)


In [135]:
# write only table, for direct use in report
with open(file_prefix + '.tex', 'w') as f:
    f.write(latex_table)